# Ruta del dinero en Bre-B

Este notebook arma la red de cuentas a partir de transferencias Bre-B y sigue el dinero en las dos direcciones:

- **De dónde viene:** las transferencias que entraron a una cuenta alertada y, si el origen también es sospechoso, de dónde le llegó a ese origen.
- **Para dónde va y a dónde sigue:** salto por salto, cuánto del dinero que entró sale hacia qué cuentas, hasta que sale de Cotrafa o se queda en una cuenta.

Todos los datos son **sintéticos**. El generador inserta cuatro tipologías conocidas (estafa con estratificación, muchos a uno, uno a muchos y ciclo) para comprobar que las reglas las encuentran.

**Límite de visibilidad.** Cotrafa solo ve una transferencia cuando el origen o el destino es una cuenta de Cotrafa. Lo que pase entre dos cuentas de otras entidades no se ve. El generador sí crea esos movimientos, pero el análisis solo usa los visibles (`visible_cotrafa`).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

SEMILLA = 2025
INICIO = pd.Timestamp("2026-11-02 00:00")  # lunes
DIAS = 7
N_COTRAFA = 400
N_EXTERNAS = 300
N_TRANSFERENCIAS = 6000

ENTIDADES_EXTERNAS = ["Entidad A", "Entidad B", "Entidad C", "Entidad D", "Entidad E"]
LLAVES = ["Celular", "Documento", "Correo", "Alfanumérica"]
MUNICIPIOS = ["Medellín", "Bello", "Itagüí", "Envigado", "Rionegro", "Marinilla", "Apartadó", "Caucasia"]
PESOS_MUNICIPIO = [0.38, 0.13, 0.11, 0.1, 0.1, 0.07, 0.06, 0.05]

# Parámetros del rastreo
VENTANA_HORAS = 48       # el dinero que entra se busca en salidas de las siguientes 48 h
MAX_SALTOS = 4           # saltos hacia adelante desde la entrada
MONTO_MINIMO = 10_000    # por debajo de esto se deja de rastrear un hilo

SALIDA = Path(".")
rng = np.random.default_rng(SEMILLA)

## 1. Datos sintéticos

Cuentas de Cotrafa (`CTF-`) y de otras entidades (`EXT-`). De las cuentas externas Cotrafa solo conoce la entidad, no la antigüedad. Las cuentas que participan en las tipologías llevan `M` en el código (`CTF-M01`) para poder evaluar el resultado al final. El análisis no usa ese dato.

In [2]:
def crear_cuentas():
    filas = []
    for i in range(1, N_COTRAFA + 1):
        nueva = rng.random() < 0.05
        filas.append({
            "cuenta": f"CTF-{i:04d}",
            "entidad": "Cotrafa",
            "tipo_llave": rng.choice(LLAVES, p=[0.5, 0.25, 0.1, 0.15]),
            "antiguedad_dias": int(rng.integers(10, 60) if nueva else rng.integers(60, 5000)),
            "municipio": rng.choice(MUNICIPIOS, p=PESOS_MUNICIPIO),
        })
    for i in range(1, N_EXTERNAS + 1):
        filas.append({
            "cuenta": f"EXT-{i:04d}",
            "entidad": rng.choice(ENTIDADES_EXTERNAS),
            "tipo_llave": rng.choice(LLAVES, p=[0.5, 0.25, 0.1, 0.15]),
            "antiguedad_dias": np.nan,
            "municipio": None,
        })
    return pd.DataFrame(filas)


def redondear(monto):
    return float(round(monto / 100) * 100)


def transferencias_normales(cuentas, n):
    ctf = cuentas.loc[cuentas.entidad == "Cotrafa", "cuenta"].to_numpy()
    ext = cuentas.loc[cuentas.entidad != "Cotrafa", "cuenta"].to_numpy()
    tipo = rng.choice(["ctf_ext", "ctf_ctf", "ext_ctf"], n, p=[0.33, 0.22, 0.45])
    origen = np.where(tipo == "ext_ctf", rng.choice(ext, n), rng.choice(ctf, n))
    destino = np.where(tipo == "ctf_ext", rng.choice(ext, n), rng.choice(ctf, n))
    dia = rng.integers(0, DIAS, n)
    hora = np.clip(rng.normal(14, 4, n), 0, 23.99)
    fecha = INICIO + pd.to_timedelta(dia, unit="D") + pd.to_timedelta(hora, unit="h")
    monto = np.round(np.exp(rng.normal(np.log(90_000), 0.9, n)) / 100) * 100
    df = pd.DataFrame({"fecha": fecha, "origen": origen, "destino": destino, "monto": monto,
                       "tipologia": "normal"})
    return df[df.origen != df.destino]


def minutos(m):
    return pd.Timedelta(minutes=int(m))


def tipologias(cuentas):
    """Inserta cuatro patrones conocidos. Devuelve transferencias y cuentas nuevas."""
    tx = []

    def add(fecha, origen, destino, monto, tipologia):
        tx.append({"fecha": fecha, "origen": origen, "destino": destino,
                   "monto": redondear(monto), "tipologia": tipologia})

    nuevas = [  # cuenta, entidad, antigüedad
        ("CTF-M01", "Cotrafa", 12), ("CTF-M02", "Cotrafa", 25), ("CTF-M03", "Cotrafa", 8),
        ("CTF-M04", "Cotrafa", 40), ("CTF-M05", "Cotrafa", 18), ("CTF-M06", "Cotrafa", 33),
        ("CTF-M07", "Cotrafa", 20),
        ("EXT-M10", "Entidad C", np.nan), ("EXT-M11", "Entidad B", np.nan), ("EXT-M12", "Entidad D", np.nan),
        ("EXT-M13", "Entidad E", np.nan), ("EXT-M14", "Entidad A", np.nan), ("EXT-M15", "Entidad C", np.nan),
        ("EXT-M16", "Entidad D", np.nan), ("EXT-M17", "Entidad B", np.nan),
    ]
    nuevas = pd.DataFrame([{"cuenta": c, "entidad": e, "tipo_llave": "Celular", "antiguedad_dias": a,
                            "municipio": "Medellín" if e == "Cotrafa" else None} for c, e, a in nuevas])

    antiguas = cuentas[(cuentas.entidad == "Cotrafa") & (cuentas.antiguedad_dias > 365)].cuenta.to_numpy()
    ext = cuentas.loc[cuentas.entidad != "Cotrafa", "cuenta"].to_numpy()

    # Caso 1. Estafa y estratificación: 5 asociados engañados transfieren a una cuenta mula,
    # que reparte en minutos a otra cuenta de Cotrafa y a otra entidad.
    t0 = INICIO + pd.Timedelta(days=2, hours=10)
    for victima in rng.choice(antiguas, 5, replace=False):
        t = t0 + minutos(rng.integers(0, 90))
        m = rng.uniform(1.5e6, 6e6)
        add(t, victima, "CTF-M01", m, "estafa_estratificacion")
        t1 = t + minutos(rng.integers(5, 25))
        add(t1, "CTF-M01", "CTF-M02", m * 0.60, "estafa_estratificacion")
        add(t1 + minutos(2), "CTF-M01", "EXT-M10", m * 0.35, "estafa_estratificacion")
        t2 = t1 + minutos(rng.integers(10, 40))
        add(t2, "CTF-M02", "EXT-M11", m * 0.60 * 0.95, "estafa_estratificacion")
        # Este movimiento ocurre entre dos entidades externas: Cotrafa no lo ve.
        add(t2 + minutos(rng.integers(5, 30)), "EXT-M11", "EXT-M12", m * 0.60 * 0.9, "estafa_estratificacion")

    # Caso 2. Muchos a uno: 14 cuentas externas envían montos pequeños en 3 horas;
    # la cuenta se vacía de madrugada hacia otra entidad.
    total = 0
    for origen in rng.choice(ext, 14, replace=False):
        m = rng.uniform(150_000, 450_000)
        total += m
        add(INICIO + pd.Timedelta(days=4, hours=19) + minutos(rng.integers(0, 180)), origen, "CTF-M03", m, "muchos_a_uno")
    add(INICIO + pd.Timedelta(days=5, hours=1, minutes=30), "CTF-M03", "EXT-M13", total * 0.97, "muchos_a_uno")

    # Caso 3. Uno a muchos: entra un monto grande y se fracciona en 11 transferencias menores a $1 millón.
    t = INICIO + pd.Timedelta(days=1, hours=15)
    add(t, "EXT-M14", "CTF-M04", 9_800_000, "uno_a_muchos")
    destinos = list(rng.choice(antiguas, 6, replace=False)) + ["CTF-M05"] + list(rng.choice(ext, 4, replace=False))
    for d in destinos:
        m = rng.uniform(700_000, 950_000)
        ts = t + minutos(rng.integers(5, 90))
        add(ts, "CTF-M04", d, m, "uno_a_muchos")
        if d == "CTF-M05":
            add(ts + minutos(rng.integers(10, 30)), "CTF-M05", "EXT-M16", m * 0.96, "uno_a_muchos")

    # Caso 4. Ciclo: el dinero sale, pasa por otra entidad y vuelve a la cuenta de partida.
    t = INICIO + pd.Timedelta(days=3, hours=10)
    add(t, "EXT-M17", "CTF-M06", 3_300_000, "ciclo")
    add(t + minutos(60), "CTF-M06", "CTF-M07", 3_200_000, "ciclo")
    add(t + minutos(100), "CTF-M07", "EXT-M15", 3_100_000, "ciclo")
    add(t + minutos(280), "EXT-M15", "CTF-M06", 3_000_000, "ciclo")

    return pd.DataFrame(tx), nuevas


cuentas = crear_cuentas()
tx_normales = transferencias_normales(cuentas, N_TRANSFERENCIAS)
tx_tipologias, cuentas_tipologias = tipologias(cuentas)
cuentas = pd.concat([cuentas, cuentas_tipologias], ignore_index=True)

transferencias = (pd.concat([tx_normales, tx_tipologias], ignore_index=True)
                  .sort_values("fecha", ignore_index=True))
transferencias.insert(0, "tx_id", [f"BRB-{i:06d}" for i in range(1, len(transferencias) + 1)])
entidad = cuentas.set_index("cuenta").entidad
transferencias["entidad_origen"] = transferencias.origen.map(entidad)
transferencias["entidad_destino"] = transferencias.destino.map(entidad)
transferencias["visible_cotrafa"] = (transferencias.entidad_origen.eq("Cotrafa")
                                     | transferencias.entidad_destino.eq("Cotrafa"))

print(f"{len(transferencias):,} transferencias, {transferencias.visible_cotrafa.sum():,} visibles para Cotrafa")
transferencias.groupby("tipologia").agg(transferencias=("tx_id", "size"), monto=("monto", "sum"))

6,052 transferencias, 6,047 visibles para Cotrafa


,transferencias,monto
tipologia,,
ciclo,4,12600000.0
estafa_estratificacion,25,50551000.0
muchos_a_uno,15,8238100.0
normal,5995,794610000.0
uno_a_muchos,13,19721600.0


## 2. Datos reales (opcional)

Para usar datos reales, cargue una tabla con estas columnas y reemplace `transferencias` y `cuentas`. Ejecute esto **solo en el entorno de Cotrafa** y no suba los datos ni las salidas al repositorio.

| Columna | Contenido |
|---|---|
| `tx_id` | identificador de la transferencia |
| `fecha` | fecha y hora |
| `origen`, `destino` | cuenta o llave seudonimizada |
| `entidad_origen`, `entidad_destino` | `Cotrafa` o el nombre de la otra entidad |
| `monto` | valor en pesos |

`cuentas` necesita `cuenta`, `entidad` y `antiguedad_dias` (las externas pueden ir vacías).

In [3]:
# transferencias = pd.read_excel("movimientos_breb.xlsx", parse_dates=["fecha"])
# cuentas = pd.read_excel("cuentas_breb.xlsx")
# transferencias["tipologia"] = "desconocida"
# transferencias["visible_cotrafa"] = True  # en datos reales todo lo que tiene Cotrafa es visible
# entidad = cuentas.set_index("cuenta").entidad

## 3. Indicadores de flujo por cuenta

Para cada cuenta de Cotrafa, con las transferencias visibles:

| Indicador | Qué mide |
|---|---|
| `ratio_paso` | salidas / entradas en pesos. Cerca de 1 significa que el dinero no se queda. |
| `permanencia_min` | mediana de minutos entre la última entrada y cada salida. |
| `max_origenes_24h` | máximo de cuentas distintas que le enviaron dentro de 24 horas. |
| `max_destinos_24h` | máximo de cuentas distintas a las que envió dentro de 24 horas. |
| `salida_madrugada` | envió $1 millón o más entre las 00:00 y las 05:00. |

In [4]:
visibles = transferencias[transferencias.visible_cotrafa].reset_index(drop=True)


def max_distintos_24h(fechas, contrapartes):
    fechas = fechas.to_numpy()
    contrapartes = contrapartes.to_numpy()
    maximo, j = 0, 0
    for i in range(len(fechas)):
        while j < len(fechas) and fechas[j] <= fechas[i] + np.timedelta64(24, "h"):
            j += 1
        maximo = max(maximo, len(set(contrapartes[i:j])))
    return maximo


def indicadores_flujo(visibles, cuentas):
    entradas = {c: g.sort_values("fecha") for c, g in visibles.groupby("destino")}
    salidas = {c: g.sort_values("fecha") for c, g in visibles.groupby("origen")}
    vacio = visibles.iloc[0:0]
    filas = []
    for fila in cuentas[cuentas.entidad == "Cotrafa"].itertuples():
        ent = entradas.get(fila.cuenta, vacio)
        sal = salidas.get(fila.cuenta, vacio)
        permanencia = np.nan
        if len(ent) and len(sal):
            t_ent = ent.fecha.to_numpy()
            pos = np.searchsorted(t_ent, sal.fecha.to_numpy(), side="right") - 1
            validos = pos >= 0
            if validos.any():
                espera = (sal.fecha.to_numpy()[validos] - t_ent[pos[validos]]) / np.timedelta64(1, "m")
                permanencia = float(np.median(espera))
        monto_ent, monto_sal = ent.monto.sum(), sal.monto.sum()
        filas.append({
            "cuenta": fila.cuenta,
            "antiguedad_dias": fila.antiguedad_dias,
            "entradas_n": len(ent), "entradas_monto": monto_ent,
            "salidas_n": len(sal), "salidas_monto": monto_sal,
            "ratio_paso": monto_sal / monto_ent if monto_ent else np.nan,
            "permanencia_min": permanencia,
            "max_origenes_24h": max_distintos_24h(ent.fecha, ent.origen) if len(ent) else 0,
            "max_destinos_24h": max_distintos_24h(sal.fecha, sal.destino) if len(sal) else 0,
            "salida_madrugada": bool(((sal.fecha.dt.hour < 5) & (sal.monto >= 1_000_000)).any()),
        })
    return pd.DataFrame(filas)


flujo = indicadores_flujo(visibles, cuentas)
flujo.describe().T[["mean", "50%", "max"]].round(1)

,mean,50%,max
antiguedad_dias,2387.6,2315.0,4988.0
entradas_n,10.1,10.0,19.0
entradas_monto,1476828.7,1261300.0,16520000.0
salidas_n,8.2,8.0,19.0
salidas_monto,1209176.2,992400.0,15693800.0
ratio_paso,1.2,0.8,61.8
permanencia_min,745.1,665.3,4699.0
max_origenes_24h,3.8,4.0,14.0
max_destinos_24h,3.3,3.0,11.0


## 4. Ciclos

Un ciclo es una cadena A → B → … → A dentro de 48 horas, con montos de $1 millón o más y parecidos entre sí (±20%). Indica que el dinero da una vuelta para aparentar movimientos legítimos.

In [5]:
def detectar_ciclos(visibles, ventana_h=VENTANA_HORAS, tolerancia=0.2, largo_max=4):
    grandes = visibles[visibles.monto >= 1_000_000].sort_values("fecha")
    salidas = {c: list(g.itertuples()) for c, g in grandes.groupby("origen")}
    ventana = pd.Timedelta(hours=ventana_h)
    ciclos = []

    def extender(camino):
        ultimo = camino[-1]
        for e in salidas.get(ultimo.destino, []):
            if e.fecha <= ultimo.fecha or e.fecha > camino[0].fecha + ventana:
                continue
            if abs(e.monto - ultimo.monto) > tolerancia * ultimo.monto:
                continue
            if e.destino == camino[0].origen:
                ciclos.append(camino + [e])
            elif len(camino) < largo_max - 1 and e.destino not in {x.origen for x in camino}:
                extender(camino + [e])

    for e in grandes.itertuples():
        extender([e])
    return [[e.tx_id for e in c] for c in ciclos]


ciclos = detectar_ciclos(visibles)
cuentas_en_ciclo = {c for ciclo in ciclos
                    for c in visibles.set_index("tx_id").loc[ciclo, ["origen", "destino"]].to_numpy().ravel()}
for ciclo in ciclos:
    print(" → ".join(visibles.set_index("tx_id").loc[ciclo, "origen"]) + " → (vuelve)")

CTF-M06 → CTF-M07 → EXT-M15 → (vuelve)


## 5. Puntaje de riesgo por cuenta

Reglas explicables con tope en 100. Los pesos son una propuesta inicial para calibrar con Riesgo y SARLAFT.

In [6]:
REGLAS = {
    "cuenta_puente": ("Cuenta puente: entra y sale en menos de 1 hora", 40),
    "muchos_a_uno": ("Muchos a uno: 8 o más orígenes en 24 h", 25),
    "uno_a_muchos": ("Uno a muchos: 8 o más destinos en 24 h", 25),
    "ciclo": ("Ciclo: el dinero vuelve a la cuenta", 30),
    "cuenta_nueva": ("Cuenta con menos de 60 días", 20),
    "salida_madrugada": ("Salida de $1 millón o más entre 00:00 y 05:00", 15),
}


def puntuar(flujo):
    f = flujo.copy()
    f["cuenta_puente"] = (f.entradas_monto >= 500_000) & f.ratio_paso.between(0.8, 1.05) & (f.permanencia_min <= 60)
    f["muchos_a_uno"] = f.max_origenes_24h >= 8
    f["uno_a_muchos"] = f.max_destinos_24h >= 8
    f["ciclo"] = f.cuenta.isin(cuentas_en_ciclo)
    f["cuenta_nueva"] = f.antiguedad_dias < 60
    f["score"] = sum(f[r].astype(int) * peso for r, (_, peso) in REGLAS.items()).clip(upper=100)
    f["nivel"] = pd.cut(f.score, [-1, 29, 59, 79, 100], labels=["bajo", "medio", "alto", "crítico"])
    f["reglas"] = f.apply(lambda x: [r for r in REGLAS if x[r]], axis=1)
    return f.sort_values("score", ascending=False, ignore_index=True)


riesgo = puntuar(flujo)
riesgo[riesgo.score >= 30][["cuenta", "score", "nivel", "reglas", "entradas_monto", "salidas_monto",
                            "permanencia_min", "max_origenes_24h", "max_destinos_24h"]]

,cuenta,score,nivel,reglas,entradas_monto,salidas_monto,permanencia_min,max_origenes_24h,max_destinos_24h
0,CTF-M07,90,crítico,"[cuenta_puente, ciclo, cuenta_nueva]",3200000.0,3100000.0,40.000000,1,1
1,CTF-M04,85,crítico,"[cuenta_puente, uno_a_muchos, cuenta_nueva]",9800000.0,9159000.0,23.000000,1,11
2,CTF-M01,60,alto,"[cuenta_puente, cuenta_nueva]",16520000.0,15693800.0,14.500000,5,2
3,CTF-M02,60,alto,"[cuenta_puente, cuenta_nueva]",9911900.0,9416300.0,16.000000,1,1
4,CTF-M03,60,alto,"[muchos_a_uno, cuenta_nueva, salida_madrugada]",4181800.0,4056300.0,211.000000,14,1
5,CTF-M05,60,alto,"[cuenta_puente, cuenta_nueva]",794400.0,762600.0,23.000000,1,1
6,CTF-M06,50,medio,"[ciclo, cuenta_nueva]",6300000.0,3200000.0,60.000000,2,1
7,CTF-0184,40,medio,[cuenta_puente],1512700.0,1252900.0,32.423894,3,3


## 6. Rastreo del dinero

**Hacia adelante (para dónde va).** Se parte de las transferencias que entraron a la cuenta alertada. Para cada cuenta de destino se buscan sus salidas en las siguientes 48 horas y se les asigna el dinero en orden de llegada (primero en entrar, primero en salir). Una salida que ya se le asignó a otro hilo no se cuenta dos veces. El rastreo sigue hasta `MAX_SALTOS` o hasta que el dinero sale de Cotrafa.

El rastreo se detiene en dos casos:

- **Llega a una cuenta de Cotrafa sin riesgo** (puntaje < 30), por ejemplo un asociado que recibió un pago. Ahí el dinero se mezcla con el saldo propio del asociado, y atribuirle sus gastos siguientes sería inventar la ruta. La cuenta queda marcada para revisión.
- **Sale hacia otra entidad.** Cotrafa solo puede seguirlo si esa cuenta externa le envía después a una cuenta de Cotrafa con riesgo (por ejemplo, en un ciclo). Si no, el rastro termina ahí.

**Hacia atrás (de dónde viene).** Desde las entradas de la cuenta alertada, si el origen es una cuenta de Cotrafa con riesgo (puntaje ≥ 30), se buscan sus entradas en las 48 horas anteriores, de la más reciente a la más antigua. Si el origen es una cuenta normal (por ejemplo, un asociado víctima), el rastreo hacia atrás se detiene: ese es el origen del dinero.

In [7]:
SCORE = riesgo.set_index("cuenta").score


def es_cotrafa(cuenta):
    return entidad.get(cuenta) == "Cotrafa"


def paso(e, salto, monto, direccion):
    return {"salto": salto, "direccion": direccion, "tx_id": e.tx_id, "fecha": e.fecha,
            "origen": e.origen, "destino": e.destino, "entidad_origen": e.entidad_origen,
            "entidad_destino": e.entidad_destino, "monto_tx": e.monto, "monto_rastreado": round(monto)}


def rastrear_destino(visibles, semillas, max_saltos=MAX_SALTOS, ventana_h=VENTANA_HORAS, minimo=MONTO_MINIMO,
                     score_min=30):
    salidas = {c: list(g.sort_values("fecha").itertuples()) for c, g in visibles.groupby("origen")}
    ventana = pd.Timedelta(hours=ventana_h)
    usado = {}
    pasos, cola = [], []
    for e in semillas.sort_values("fecha").itertuples():
        pasos.append(paso(e, 1, e.monto, "entrada"))
        cola.append((e, e.monto, 1))
    while cola:
        e, monto, salto = cola.pop(0)
        cotrafa = es_cotrafa(e.destino)
        if salto >= max_saltos or (cotrafa and SCORE.get(e.destino, 0) < score_min):
            continue
        restante = monto
        for s in salidas.get(e.destino, []):
            if s.fecha <= e.fecha or s.fecha > e.fecha + ventana:
                continue
            if not cotrafa and SCORE.get(s.destino, 0) < score_min:
                continue  # desde otra entidad solo se sigue si vuelve a una cuenta de riesgo
            disponible = s.monto - usado.get(s.tx_id, 0)
            if disponible <= 0:
                continue
            tomado = min(disponible, restante)
            usado[s.tx_id] = usado.get(s.tx_id, 0) + tomado
            restante -= tomado
            pasos.append(paso(s, salto + 1, tomado, "adelante"))
            cola.append((s, tomado, salto + 1))
            if restante < minimo:
                break
    return pd.DataFrame(pasos)


def rastrear_origen(visibles, semillas, max_saltos=2, ventana_h=VENTANA_HORAS, minimo=MONTO_MINIMO, score_min=30):
    entradas = {c: list(g.sort_values("fecha", ascending=False).itertuples()) for c, g in visibles.groupby("destino")}
    ventana = pd.Timedelta(hours=ventana_h)
    usado = {}
    pasos = []
    cola = [(e, e.monto, 1) for e in semillas.itertuples()]
    while cola:
        e, monto, salto = cola.pop(0)
        if salto - 1 >= max_saltos or not es_cotrafa(e.origen) or SCORE.get(e.origen, 0) < score_min:
            continue
        restante = monto
        for s in entradas.get(e.origen, []):
            if s.fecha >= e.fecha or s.fecha < e.fecha - ventana:
                continue
            disponible = s.monto - usado.get(s.tx_id, 0)
            if disponible <= 0:
                continue
            tomado = min(disponible, restante)
            usado[s.tx_id] = usado.get(s.tx_id, 0) + tomado
            restante -= tomado
            pasos.append(paso(s, salto - 1, tomado, "atras"))
            cola.append((s, tomado, salto - 1))
            if restante < minimo:
                break
    return pd.DataFrame(pasos)


def rastrear_cuenta(cuenta, max_saltos=MAX_SALTOS):
    semillas = visibles[visibles.destino == cuenta]
    ruta = pd.concat([rastrear_origen(visibles, semillas), rastrear_destino(visibles, semillas, max_saltos)],
                     ignore_index=True)
    return (ruta.groupby("tx_id", as_index=False)
                .agg(salto=("salto", "min"), direccion=("direccion", "first"), fecha=("fecha", "first"),
                     origen=("origen", "first"), destino=("destino", "first"),
                     entidad_origen=("entidad_origen", "first"), entidad_destino=("entidad_destino", "first"),
                     monto_tx=("monto_tx", "first"), monto_rastreado=("monto_rastreado", "sum"))
                .sort_values(["salto", "fecha"], ignore_index=True))


ejemplo = riesgo.cuenta.iloc[0]
print("Cuenta con mayor riesgo:", ejemplo)
rastrear_cuenta(ejemplo)

Cuenta con mayor riesgo: CTF-M07


,tx_id,salto,direccion,fecha,origen,destino,entidad_origen,entidad_destino,monto_tx,monto_rastreado
0,BRB-002754,0,atras,2026-11-05 10:00:00,EXT-M17,CTF-M06,Entidad B,Cotrafa,3300000.0,3200000
1,BRB-002828,1,entrada,2026-11-05 11:00:00,CTF-M06,CTF-M07,Cotrafa,Cotrafa,3200000.0,3200000
2,BRB-002881,2,adelante,2026-11-05 11:40:00,CTF-M07,EXT-M15,Cotrafa,Entidad C,3100000.0,3100000
3,BRB-003154,3,adelante,2026-11-05 14:40:00,EXT-M15,CTF-M06,Entidad C,Cotrafa,3000000.0,3000000


## 7. Resumen de cada ruta

Para cada ruta: cuánto entró a la cuenta alertada, cuánto salió de Cotrafa y hacia qué entidades, y cuánto quedó en cuentas de Cotrafa (donde todavía se puede congelar o recuperar).

In [8]:
def resumir_ruta(cuenta, ruta):
    entrada = ruta[(ruta.destino == cuenta) & (ruta.direccion == "entrada")].monto_rastreado.sum()
    adelante = ruta[ruta.direccion != "atras"]
    # Lo que llega a cada cuenta externa, menos lo que esa cuenta devuelve a Cotrafa dentro de la ruta.
    llega = adelante[adelante.entidad_destino != "Cotrafa"].groupby(["destino", "entidad_destino"]).monto_rastreado.sum()
    devuelve = adelante[adelante.entidad_origen != "Cotrafa"].groupby("origen").monto_rastreado.sum()
    neto = (llega - devuelve.reindex(llega.index.get_level_values(0)).fillna(0).to_numpy()).clip(lower=0)
    salio = neto.sum()
    por_entidad = neto.groupby(level=1).sum().sort_values(ascending=False)
    return {"entro": float(entrada), "salio_cotrafa": float(salio),
            "queda_en_cotrafa": float(max(entrada - salio, 0)),
            "salio_por_entidad": {k: float(v) for k, v in por_entidad.items() if v > 0}}


alertadas = riesgo[riesgo.score >= 60].cuenta.tolist()
for c in alertadas:
    r = resumir_ruta(c, rastrear_cuenta(c))
    print(f"{c}: entró ${r['entro']:,.0f} · salió de Cotrafa ${r['salio_cotrafa']:,.0f} {r['salio_por_entidad']}")

CTF-M07: entró $3,200,000 · salió de Cotrafa $100,000 {'Entidad C': 100000.0}
CTF-M04: entró $9,800,000 · salió de Cotrafa $4,060,000 {'Entidad A': 1565600.0, 'Entidad C': 869800.0, 'Entidad B': 862000.0, 'Entidad D': 762600.0}
CTF-M01: entró $16,520,000 · salió de Cotrafa $15,198,200 {'Entidad B': 9416300.0, 'Entidad C': 5781900.0}
CTF-M02: entró $9,911,900 · salió de Cotrafa $9,416,300 {'Entidad B': 9416300.0}
CTF-M03: entró $4,181,800 · salió de Cotrafa $4,056,300 {'Entidad E': 4056300.0}
CTF-M05: entró $794,400 · salió de Cotrafa $762,600 {'Entidad D': 762600.0}


## 8. Evaluación contra las tipologías insertadas

Solo es posible con datos sintéticos: se sabe cuáles cuentas participan en cada tipología.

In [9]:
mulas = set(cuentas[cuentas.cuenta.str.startswith("CTF-M")].cuenta)
alerta = set(riesgo[riesgo.score >= 60].cuenta)
print(f"Cuentas mula insertadas: {len(mulas)}")
print(f"Alertas (score ≥ 60): {len(alerta)}")
print(f"Mulas detectadas: {len(mulas & alerta)} · no detectadas: {sorted(mulas - alerta)}")
print(f"Falsos positivos: {sorted(alerta - mulas)}")

Cuentas mula insertadas: 7
Alertas (score ≥ 60): 6
Mulas detectadas: 6 · no detectadas: ['CTF-M06']
Falsos positivos: []


## 9. Exportar

- `breb_movimientos_sinteticos.csv`: todas las transferencias, con `visible_cotrafa` y la tipología (esta última solo sirve para evaluar).
- `breb_cuentas_riesgo.csv`: indicadores, reglas y puntaje por cuenta.
- `ruta_dinero.json`: los casos que muestra el diagrama de `bre-b-sentinel.html`.

Cada caso agrupa una cuenta alertada con las cuentas que aparecen en su ruta; una cuenta que ya salió en un caso no abre otro.

In [10]:
TITULOS = {
    "cuenta_puente": "Cuenta puente", "muchos_a_uno": "Muchos a uno", "uno_a_muchos": "Uno a muchos",
    "ciclo": "Ciclo", "salida_madrugada": "Salida de madrugada", "cuenta_nueva": "Cuenta nueva",
}


def caso_json(cuenta, max_saltos=3):
    ruta = rastrear_cuenta(cuenta, max_saltos)
    fila = riesgo.set_index("cuenta").loc[cuenta]
    columnas = {}
    for e in ruta.sort_values(["salto", "fecha"]).itertuples():
        # salto k: el origen va en la columna k-1 y el destino en la k (las entradas tienen salto 1)
        columnas.setdefault(e.origen, e.salto - 1)
        columnas.setdefault(e.destino, e.salto)
    base = min(columnas.values())
    externas_con_salida = set(ruta[ruta.entidad_origen != "Cotrafa"].origen)
    nodos = [{
        "id": c, "entidad": entidad.get(c), "col": col - base,
        "score": int(SCORE.get(c, 0)) if es_cotrafa(c) else None,
        "tipo": ("alertada" if c == cuenta else
                 "riesgo" if es_cotrafa(c) and SCORE.get(c, 0) >= 30 else
                 "cotrafa" if es_cotrafa(c) else
                 "externa" if c in externas_con_salida else "sale"),
    } for c, col in columnas.items()]
    reglas = [TITULOS[r] for r in fila.reglas if r != "cuenta_nueva"]
    return {
        "cuenta": cuenta, "score": int(fila.score), "nivel": str(fila.nivel),
        "titulo": " + ".join(reglas) or "Cuenta nueva",
        "reglas": [REGLAS[r][0] for r in fila.reglas],
        "resumen": resumir_ruta(cuenta, ruta),
        "nodos": nodos,
        "enlaces": [{"tx": e.tx_id, "de": e.origen, "a": e.destino, "salto": int(e.salto - base),
                     "fecha": e.fecha.strftime("%Y-%m-%d %H:%M"), "monto_tx": e.monto_tx,
                     "monto": float(e.monto_rastreado)} for e in ruta.itertuples()],
    }


casos, cubiertas = [], set()
for c in riesgo[riesgo.score >= 60].cuenta:
    if c in cubiertas:
        continue
    caso = caso_json(c)
    casos.append(caso)
    cubiertas |= {n["id"] for n in caso["nodos"]}
    if len(casos) == 4:
        break

transferencias.to_csv(SALIDA / "breb_movimientos_sinteticos.csv", index=False)
riesgo.assign(reglas=riesgo.reglas.str.join("|")).to_csv(SALIDA / "breb_cuentas_riesgo.csv", index=False)
(SALIDA / "ruta_dinero.json").write_text(
    json.dumps({"semilla": SEMILLA, "ventana_horas": VENTANA_HORAS, "casos": casos}, ensure_ascii=False, indent=1),
    encoding="utf-8")
print(f"{len(casos)} casos exportados:", [(c["cuenta"], c["titulo"]) for c in casos])

4 casos exportados: [('CTF-M07', 'Cuenta puente + Ciclo'), ('CTF-M04', 'Cuenta puente + Uno a muchos'), ('CTF-M01', 'Cuenta puente'), ('CTF-M03', 'Muchos a uno + Salida de madrugada')]
